## RNN(순환 신경망)
- 입력과 출력을 시퀀스 단위로 처리하는 시퀀스(Sequence) 모델
- 시퀀스 데이터는 단어, 음성, 영상 등과 같이 순서가 있는 데이터
- 피드 포워드 신경망(Feedforward Neural Network)은 입력과 출력이 고정된 크기를 가지며, 시퀀스 데이터를 처리하기 어려움
- 은닉층의 노드에서 활성화 함수를 통해 나온 결과값을 출력층으로 보내면서 다시 은닉층 노드의 다음 계산의 입력으로 보냄.
- 이런 역할을 하는 노드를 **셀(cell)** 이라고 부름
- 이전의 값을 기억하므로 **메모리 셀** 또는 **RNN 셀** 이라고도 불림
- 메모리 셀이 출력층 방향 또는 다음 시점인 $t+1$ 의 자신에게 보내는 값을 **은닉 상태(hidden state)** 라고 함.
- RNN 셀은 시점 $t$ 의 입력과 시점 $t-1$ 의 은닉 상태를 받아서 시점 $t$ 의 은닉 상태를 계산
- <img src="img/zzf.png" width="400">

- RNN은 입력과 출력 의 길이를 다르게 설계할 수 있음.(보편적으로 사용하는 입출력의 단위는 '단어 벡터')
### 일 대 다(One-to-Many)
- 입력이 하나이고 출력이 여러 개인 경우
- 예시:
    - 이미지 캡셔닝(Image Captioning): 하나의 이미지 입력에 대해 제목을 출력
        - 입력(이미지) -> 출력('A', 'cat', 'sitting', 'on', 'a', 'mat.')

### 다 대 일(Many-to-One)
- 입력이 여러 개이고 출력이 하나인 경우
- 예시:
    - 감성 분석(Sentiment Analysis): 여러 단어로 이루어진 문장 입력에 대해 긍정/부정 판별
    - 스팸 메일 분류(Spam Email Classification): 여러 단어로 이루어진 이메일 입력에 대해 스팸/비스팸 판별
        - 입력('claim', 'your', 'winning', 'prize') -> 출력('스팸')

### 다 대 다(Many-to-Many)
- 입력과 출력이 모두 여러 개인 경우
- 예시:
    - 기계 번역(Machine Translation): 여러 단어로 이루어진 문장 입력에 대해 다른 언어로 번역된 문장 출력
    - 챗봇(Chatbot): 여러 단어로 이루어진 질문 입력에 대해 여러 단어로 이루어진 답변 출력
        - 입력('What', 'is', 'your', 'name?') -> 출력('My', 'name', 'is', 'Chatbot.')

### 가중치
- RNN은 입력 $x$에 대한 가중치 $w_{x}$, 은닉 상태 h에 대한 가중치 $w_{h}$, 출력 y에 대한 가중치 $w_{y}$ 를 가짐
- $h_{t} = \tanh(w_{x} x_{t} + w_{h} h_{t-1} + b)$ (은닉 상태 계산)
- $y_{t} = f(w_{y} h_{t} + b_y)$ ($f$는 비선형 함수, 예: 소프트맥스)

### Keras로 RNN 모델 구축
1. `model.add(SimpleRNN(hidden_units, input_shape=(timesteps, input_dim)))`
- hidden_units: 은닉 상태의 크기 정의. 다음 시점의 메모리 셀과 출력층으로 보내는 값의 크기(output_dim과 동일)
- input_shape = (timesteps, input_dim):
    - (입력 시퀀스의 길이(시점(timesteps)의 수), 입력의 크기)
    - input_length=timesteps, input_dim=input_dim 로 따로 설정 가능
    - batch_shape=(None, timesteps, input_dim) 과 동일하다
---
2. `model.add(SimpleRNN(hidden_units, batch_input_shape=(8, timesteps, input_dim)))`
- batch_input_shape=(batch_size, timesteps, input_dim) 으로 배치 크기까지 명시적으로 설정 가능


- RNN 셀의 출력은 설정에 따라 다름.
- `return_sequences=False` (기본값): 마지막 시점의 은닉 상태만 출력. 출력은 (batch_size, output_dim) 로 2D 텐서
- `return_sequences=True`: 모든 시점의 은닉 상태를 출력. 출력은 (batch_size, timesteps, output_dim)로 3D 텐서
    - 이 때 output_dim은 hidden_units과 동일하다


In [8]:
# 케라스로 RNN 층 추가
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.models import Sequential

model = Sequential()
model.add(SimpleRNN(3, input_shape=(2, 10)))
model.summary()

Model: "sequential_7"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
simple_rnn_7 (SimpleRNN)     (None, 3)                 42        
Total params: 42
Trainable params: 42
Non-trainable params: 0
_________________________________________________________________


In [9]:
model1 = Sequential()
model1.add(SimpleRNN(3, batch_input_shape=(8, 2, 10)))
model1.summary()

Model: "sequential_8"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
simple_rnn_8 (SimpleRNN)     (8, 3)                    42        
Total params: 42
Trainable params: 42
Non-trainable params: 0
_________________________________________________________________


In [11]:
model2 = Sequential()
model2.add(SimpleRNN(3, batch_input_shape=(8, 2, 10), return_sequences=True))
model2.summary()

Model: "sequential_10"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
simple_rnn_10 (SimpleRNN)    (8, 2, 3)                 42        
Total params: 42
Trainable params: 42
Non-trainable params: 0
_________________________________________________________________


In [13]:
# 깊은 순환 신경망

# 은닉층이 2개인 깊은 순환 신경망
model3 = Sequential()
model3.add(SimpleRNN(3, input_shape=(10, 5), return_sequences=True))
model3.add(SimpleRNN(3, return_sequences=True))
model3.summary()

Model: "sequential_12"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
simple_rnn_13 (SimpleRNN)    (None, 10, 3)             27        
_________________________________________________________________
simple_rnn_14 (SimpleRNN)    (None, 10, 3)             21        
Total params: 48
Trainable params: 48
Non-trainable params: 0
_________________________________________________________________


## 양방향 순환 신경망(Bidirectional RNN)
- 시점 t에서의 출력값을 예측할 때 이전 시점의 입력뿐 아니라 이후 시점의 입력도 예측에 기여 가능
- ex. 영어 문장 'I am a student'에서 시점 t=2('a')의 출력값을 예측할 때 이전 시점의 입력('I', 'am')뿐 아니라 이후 시점의 입력('student')도 예측에 기여 가능
- 하나의 출력값을 예측하기 위해 기본적으로 두개의 메모리 셀을 사용한다.
    - 첫번째 메모리 셀은 앞 시점의 은닉 상태(Forward States)를 전달받아 현재 은닉 상태를 계산
    - 두번째 메모리 셀은 뒤 시점의 은닉 상태(Backward States)를 전달받아 현재 은닉 상태를 계산
    - <img src="img/zzg.png" width="400">

In [14]:
# 양방향 순환 신경망
from tensorflow.keras.layers import Bidirectional

hidden_units = 3
timesteps = 10
input_dim = 5

model = Sequential()
model.add(Bidirectional(SimpleRNN(hidden_units, return_sequences=True), input_shape=(timesteps, input_dim)))
model.add(Bidirectional(SimpleRNN(hidden_units, return_sequences=True)))
model.add(Bidirectional(SimpleRNN(hidden_units, return_sequences=True)))
model.add(Bidirectional(SimpleRNN(hidden_units, return_sequences=True)))
model.summary()

Model: "sequential_13"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
bidirectional (Bidirectional (None, 10, 6)             54        
_________________________________________________________________
bidirectional_1 (Bidirection (None, 10, 6)             60        
_________________________________________________________________
bidirectional_2 (Bidirection (None, 10, 6)             60        
_________________________________________________________________
bidirectional_3 (Bidirection (None, 10, 6)             60        
Total params: 234
Trainable params: 234
Non-trainable params: 0
_________________________________________________________________
